# Hyperparameter Tuning with Keras Tuner

In this notebook, we will learn how to perform **hyperparameter tuning** for a Neural Network using the **Keras Tuner** library.

We will tune:
1. **Optimizer** selection (Adam, SGD, RMSprop, Adadelta)
2. **Number of neurons** in a layer
3. **Number of layers** in the network
4. **Dropout rate** per layer
5. **All-in-one** combined tuning

**Dataset:** Pima Indians Diabetes Dataset (binary classification)

## Step 0: Install Keras Tuner

In [36]:
# !pip install keras-tuner

## Step 1: Import Libraries

In [37]:
import numpy as np
import pandas as pd

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

from sklearn import preprocessing
from sklearn.model_selection import train_test_split

import keras_tuner as kt

## Step 2: Load and Explore the Dataset

We are using the **Pima Indians Diabetes Dataset**. Download it from Kaggle and upload it to your environment.

- **Features:** Pregnancies, Glucose, BloodPressure, SkinThickness, Insulin, BMI, DiabetesPedigreeFunction, Age
- **Target:** Outcome (0 = No Diabetes, 1 = Diabetes)

In [38]:
df = pd.read_csv('diabetes.csv')
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [39]:
df.shape

(768, 9)

In [40]:
# Check correlation with Outcome
df.corr()['Outcome']

Pregnancies                 0.221898
Glucose                     0.466581
BloodPressure               0.065068
SkinThickness               0.074752
Insulin                     0.130548
BMI                         0.292695
DiabetesPedigreeFunction    0.173844
Age                         0.238356
Outcome                     1.000000
Name: Outcome, dtype: float64

## Step 3: Data Preprocessing

In [41]:
# Separate features and target
X = df.iloc[:, 0:-1]   # All rows, all columns except last
y = df.iloc[:, -1]     # All rows, last column only

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (768, 8)
y shape: (768,)


In [42]:
# Scale the features using StandardScaler
scaler = preprocessing.StandardScaler()
X = scaler.fit_transform(X)

print("Scaled X sample:\n", X[:3])

Scaled X sample:
 [[ 0.63994726  0.84832379  0.14964075  0.90726993 -0.69289057  0.20401277
   0.46849198  1.4259954 ]
 [-0.84488505 -1.12339636 -0.16054575  0.53090156 -0.69289057 -0.68442195
  -0.36506078 -0.19067191]
 [ 1.23388019  1.94372388 -0.26394125 -1.28821221 -0.69289057 -1.10325546
   0.60439732 -0.10558415]]


In [43]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training set size:", X_train.shape)
print("Test set size:", X_test.shape)

Training set size: (614, 8)
Test set size: (154, 8)


## Step 4: Build a Baseline Neural Network (Without Tuning)

Before hyperparameter tuning, let's build a simple model to understand the problem.

In [44]:
# Simple baseline model
model = Sequential()

model.add(Dense(32, activation='relu', input_dim=8))  # 8 input features
model.add(Dense(1, activation='sigmoid'))             # Binary output

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

c:\Users\HP\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_4 (Dense)                 │ (None, 32)             │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 321 (1.25 KB)

 Trainable params: 321 (1.25 KB)

 Non-trainable params: 0 (0.00 B)

In [45]:
# Train baseline model
history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=32,
    validation_data=(X_test, y_test)
)

Epoch 1/10


20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.6710 - loss: 0.6384 - val_accuracy: 0.6688 - val_loss: 0.6341
Epoch 2/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6971 - loss: 0.5950 - val_accuracy: 0.6883 - val_loss: 0.6082
Epoch 3/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7150 - loss: 0.5645 - val_accuracy: 0.7013 - val_loss: 0.5876
Epoch 4/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7248 - loss: 0.5408 - val_accuracy: 0.7078 - val_loss: 0.5720
Epoch 5/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7329 - loss: 0.5238 - val_accuracy: 0.7143 - val_loss: 0.5597
Epoch 6/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7362 - loss: 0.5098 - val_accuracy: 0.7273 - val_loss: 0.5511
Epoch 7/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7492 - loss: 0.4995 - val_accuracy: 0.7273 - val_loss: 0.5461
Epoch 8/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7590 - loss: 0.4903 - val_accuracy: 0.7338 - val_loss: 0.5414
Ep

## Step 5: The Problem with Manual Tuning

When building a neural network, we have to decide:
- How many hidden layers to use?
- How many neurons per layer?
- Which activation function?
- What batch size?
- Which optimizer?

We can **automate** this process using **Keras Tuner** — a library that searches for the best hyperparameters automatically.

---

## Step 6: Hyperparameter Tuning with Keras Tuner

### 6a. Tune the Optimizer

In [46]:
def build_model(hp):
    """
    Model-building function for Keras Tuner.
    hp = HyperParameters object (provided automatically by the tuner)
    """
    model = Sequential()

    # Fixed architecture — only tuning the optimizer here
    model.add(Dense(32, activation='relu', input_dim=8))
    model.add(Dense(1, activation='sigmoid'))

    # Let the tuner choose the best optimizer
    optimizer = hp.Choice(
        name='optimizer',
        values=['adam', 'sgd', 'rmsprop', 'adadelta']
    )

    model.compile(
        optimizer=optimizer,
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

In [47]:
# Create the Keras Tuner object using RandomSearch
tuner = kt.RandomSearch(
    build_model,                        # Model-building function
    objective='val_accuracy',           # Metric to optimize
    max_trials=5,                       # Number of different configs to try
    directory='my_dir',                 # Directory to store results
    project_name='optimizer_tuning'     # Project name
)

Reloading Tuner from my_dir\optimizer_tuning\tuner0.json


In [48]:
# Run the hyperparameter search
tuner.search(
    X_train, y_train,
    epochs=10,
    validation_data=(X_test, y_test)
)

In [49]:
# Get the best hyperparameters found
best_hps = tuner.get_best_hyperparameters()[0]
print("Best optimizer:", best_hps.get('optimizer'))

Best optimizer: adam


In [50]:
# Retrieve and retrain the best model
model = tuner.get_best_models(num_models=1)[0]
model.summary()

c:\Users\HP\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\saving\saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 321 (1.25 KB)

 Trainable params: 321 (1.25 KB)

 Non-trainable params: 0 (0.00 B)

In [51]:
# Continue training the best model from where search left off
model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    initial_epoch=5,       # Continue from epoch 5 (already ran 5 during search)
    validation_data=(X_test, y_test)
)

Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.7557 - loss: 0.4983 - val_accuracy: 0.7987 - val_loss: 0.4837
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7638 - loss: 0.4879 - val_accuracy: 0.7922 - val_loss: 0.4809
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7638 - loss: 0.4813 - val_accuracy: 0.7922 - val_loss: 0.4786
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7655 - loss: 0.4752 - val_accuracy: 0.7922 - val_loss: 0.4778
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7704 - loss: 0.4705 - val_accuracy: 0.7922 - val_loss: 0.4779
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7720 - loss: 0.4656 - val_accuracy: 0.7792 - val_loss: 0.4779
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7736 - loss: 0.4619 - val_accuracy: 0.7792 - val_loss: 0.4785
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7736 - loss: 0.4588 - val_accuracy: 0.779

---

### 6b. Tune the Number of Neurons per Layer

We will try different neuron counts: **8, 16, 24, 32, ... up to 128** (step = 8)

In [73]:
def build_model_neurons(hp):
    model = Sequential()

    # Tune the number of units in the first Dense layer
    units = hp.Int(
        name='units',
        min_value=8,
        max_value=128,
        step=8
    )

    model.add(Dense(units, activation='relu', input_dim=8))
    model.add(Dense(1, activation='sigmoid'))

    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

In [74]:
tuner_neurons = kt.RandomSearch(
    build_model_neurons,
    objective='val_accuracy',
    max_trials=5,
    directory='my_dir',
    project_name='neuron_tuning'
)

Reloading Tuner from my_dir\neuron_tuning\tuner0.json


In [75]:
tuner_neurons.search(
    X_train, y_train,
    epochs=10,
    validation_data=(X_test, y_test)
)

In [76]:
best_hps_neurons = tuner_neurons.get_best_hyperparameters()[0]
print("Best number of units:", best_hps_neurons.get('units'))

Best number of units: 16


---

### 6c. Tune the Number of Layers

We use a `for` loop inside the build function to dynamically add layers based on the `num_layers` hyperparameter.

In [56]:
def build_model_layers(hp):
    model = Sequential()

    # Tune number of layers
    num_layers = hp.Int(
        name='num_layers',
        min_value=1,
        max_value=10
    )

    counter = 0
    for i in range(num_layers):
        # Each layer has 72 neurons (fixed for this tuning step)
        if counter == 0:
            # First layer needs input_dim
            model.add(Dense(
                units=hp.Int(
                    name='units_' + str(i),
                    min_value=8,
                    max_value=128,
                    step=8
                ),
                activation='relu',
                input_dim=8
            ))
        else:
            model.add(Dense(
                units=hp.Int(
                    name='units_' + str(i),
                    min_value=8,
                    max_value=128,
                    step=8
                ),
                activation='relu'
            ))
        counter += 1

    # Output layer
    model.add(Dense(1, activation='sigmoid'))

    model.compile(
        optimizer='rmsprop',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

In [57]:
tuner_layers = kt.RandomSearch(
    build_model_layers,
    objective='val_accuracy',
    max_trials=5,
    directory='my_dir',
    project_name='layer_tuning'
)

Reloading Tuner from my_dir\layer_tuning\tuner0.json


In [58]:
tuner_layers.search(
    X_train, y_train,
    epochs=10,
    validation_data=(X_test, y_test)
)

In [59]:
best_hps_layers = tuner_layers.get_best_hyperparameters()[0]
print("Best number of layers:", best_hps_layers.get('num_layers'))

Best number of layers: 10


---

### 6d. Tune Dropout Rate

We add a Dropout layer after each Dense layer and tune the dropout rate.

In [60]:
def build_model_dropout(hp):
    model = Sequential()

    num_layers = hp.Int(
        name='num_layers',
        min_value=1,
        max_value=10
    )

    counter = 0
    for i in range(num_layers):
        if counter == 0:
            model.add(Dense(
                units=hp.Int('units_' + str(i), min_value=8, max_value=128, step=8),
                activation='relu',
                input_dim=8
            ))
        else:
            model.add(Dense(
                units=hp.Int('units_' + str(i), min_value=8, max_value=128, step=8),
                activation='relu'
            ))

        # Add Dropout after each layer with tunable rate
        model.add(Dropout(
            rate=hp.Choice(
                name='dropout_' + str(i),
                values=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
            )
        ))

        counter += 1

    # Output layer
    model.add(Dense(1, activation='sigmoid'))

    # Also tune the optimizer
    optimizer = hp.Choice(
        name='optimizer',
        values=['adam', 'rmsprop', 'sgd', 'adadelta']
    )

    model.compile(
        optimizer=optimizer,
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

In [61]:
tuner_dropout = kt.RandomSearch(
    build_model_dropout,
    objective='val_accuracy',
    max_trials=5,
    directory='my_dir',
    project_name='dropout_tuning'
)

Reloading Tuner from my_dir\dropout_tuning\tuner0.json


In [62]:
tuner_dropout.search(
    X_train, y_train,
    epochs=10,
    validation_data=(X_test, y_test)
)

In [63]:
best_hps_dropout = tuner_dropout.get_best_hyperparameters()[0]
print("Best hyperparameters (with dropout):")
print("  Optimizer:", best_hps_dropout.get('optimizer'))
print("  Num Layers:", best_hps_dropout.get('num_layers'))

Best hyperparameters (with dropout):
  Optimizer: sgd
  Num Layers: 10


---

## Step 7: All-in-One Model — Tune Everything Together

Now we combine all the above: tuning **optimizer**, **number of layers**, **number of neurons**, and **dropout rate** simultaneously.

In [64]:
def build_model_all(hp):
    model = Sequential()

    # Tune number of layers
    num_layers = hp.Int(
        name='num_layers',
        min_value=1,
        max_value=10
    )

    counter = 0
    for i in range(num_layers):
        if counter == 0:
            model.add(Dense(
                units=hp.Int('units_' + str(i), min_value=8, max_value=128, step=8),
                activation=hp.Choice('activation_' + str(i), values=['relu', 'tanh']),
                input_dim=8
            ))
        else:
            model.add(Dense(
                units=hp.Int('units_' + str(i), min_value=8, max_value=128, step=8),
                activation=hp.Choice('activation_' + str(i), values=['relu', 'tanh'])
            ))

        # Tune dropout rate per layer
        model.add(Dropout(
            rate=hp.Choice(
                name='dropout_' + str(i),
                values=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
            )
        ))

        counter += 1

    # Output layer
    model.add(Dense(1, activation='sigmoid'))

    # Tune optimizer
    optimizer = hp.Choice(
        name='optimizer',
        values=['adam', 'rmsprop', 'sgd', 'adadelta']
    )

    model.compile(
        optimizer=optimizer,
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

In [65]:
tuner_final = kt.RandomSearch(
    build_model_all,
    objective='val_accuracy',
    max_trials=20,
    directory='my_dir',
    project_name='final_all_in_one'
)

Reloading Tuner from my_dir\final_all_in_one\tuner0.json


In [66]:
tuner_final.search(
    X_train, y_train,
    epochs=50,
    validation_data=(X_test, y_test)
)

Trial 20 Complete [00h 00m 11s]
val_accuracy: 0.6428571343421936

Best val_accuracy So Far: 0.8116883039474487
Total elapsed time: 00h 17m 12s


In [67]:
# Get best hyperparameters
best_hps_all = tuner_final.get_best_hyperparameters()[0]
print("Best overall hyperparameters:")
print("  Optimizer:", best_hps_all.get('optimizer'))
print("  Num Layers:", best_hps_all.get('num_layers'))

Best overall hyperparameters:
  Optimizer: adam
  Num Layers: 5


In [77]:
# Get the best model
best_model = tuner_final.get_best_models(num_models=1)[0]
best_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 120)            │         1,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 120)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 16)             │         1,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │         2,176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 8)              │         1,032 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 8)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 88)             │           792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 88)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            89 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,105 (27.75 KB)

 Trainable params: 7,105 (27.75 KB)

 Non-trainable params: 0 (0.00 B)

## Step 8: Train the Best Model Fully

In [69]:
# Retrain the best model for more epochs
best_model.fit(
    X_train, y_train,
    epochs=150,
    batch_size=32,
    initial_epoch=10,   # Continue from where search stopped
    validation_data=(X_test, y_test)
)

Epoch 11/150
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.7182 - loss: 0.5682 - val_accuracy: 0.8052 - val_loss: 0.4986
Epoch 12/150
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7215 - loss: 0.5401 - val_accuracy: 0.8052 - val_loss: 0.4966
Epoch 13/150
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7134 - loss: 0.5732 - val_accuracy: 0.8052 - val_loss: 0.4983
Epoch 14/150
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7264 - loss: 0.5707 - val_accuracy: 0.8052 - val_loss: 0.4986
Epoch 15/150
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7003 - loss: 0.5380 - val_accuracy: 0.7987 - val_loss: 0.4935
Epoch 16/150
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7313 - loss: 0.5634 - val_accuracy: 0.7922 - val_loss: 0.4954
Epoch 17/150
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7231 - loss: 0.5237 - val_accuracy: 0.7792 - val_loss: 0.5003
Epoch 18/150
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7117 - loss: 0.5666 - val_accuracy: 0

## Step 9: Evaluate the Final Model

In [70]:
# Evaluate on test set
loss, accuracy = best_model.evaluate(X_test, y_test)
print(f"Test Loss:     {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7922 - loss: 0.5158 
Test Loss:     0.5158
Test Accuracy: 0.7922


---

## Summary

In this notebook, we learned how to use **Keras Tuner** for hyperparameter tuning:

| What We Tuned | Keras Tuner Method | Example |
|---|---|---|
| Optimizer | `hp.Choice()` | `['adam', 'sgd', 'rmsprop']` |
| Number of neurons | `hp.Int()` | `min=8, max=128, step=8` |
| Number of layers | `hp.Int()` | `min=1, max=10` |
| Dropout rate | `hp.Choice()` | `[0.1, 0.2, ..., 0.9]` |
| Activation function | `hp.Choice()` | `['relu', 'tanh']` |

### Keras Tuner Workflow:
1. Define a `build_model(hp)` function
2. Create a tuner: `kt.RandomSearch(build_model, objective='val_accuracy', max_trials=N)`
3. Run the search: `tuner.search(X_train, y_train, ...)`
4. Get best params: `tuner.get_best_hyperparameters()[0]`
5. Get best model: `tuner.get_best_models(num_models=1)[0]`
6. Retrain and evaluate the best model